# Lab 1.2. Object and field: from the point to the zonal statistic

**Module 2: Session 1. ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

By the end of this notebook you should be able to:

1. Read a vector layer and a raster surface, and identify in each case the data model (object or field) it follows.
2. Diagnose and repair the loss of a shapefile's reference system.
3. Justify why areas and distances are not measured in geographic coordinates, and reproject.
4. Turn a table with coordinates into a geographic table, and diagnose a CRS mismatch between two layers.
5. Sample a surface at points and aggregate it by polygons (zonal statistics).

**Before handing in:** Kernel > Restart & Run All.

## 0. Environment

In [ ]:
import sys
import pandas as pd, numpy as np, geopandas as gpd, rasterio, matplotlib
print("Python    ", sys.version.split()[0])
for m in (pd, np, gpd, rasterio, matplotlib):
    print(f"{m.__name__:<10}", m.__version__)

## 0. Setup

This cell locates the course folder wherever the notebook is running: on your own machine, in
Colab from the course repository, or in Colab from the shared Drive folder. Run it first and
check that the file listing appears.

In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                             # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):                   # local clone
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")

    root = Path("/content/acs-mod2")                   # 1. try the repository
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root

    from google.colab import drive                     # 2. fall back to Drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError(
        "Could not find the course folder. Either set REPO to the course repository, "
        f"or add a shortcut to the shared folder in My Drive as {DRIVE}.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
print("Course folder:", BASE)
print("Working folder:", WORK)
sorted(p.name for p in DATA.iterdir())

`top10.parquet` comes from exercise 6 of Lab 1.1. A Colab session is discarded after a while,
so the cell below rebuilds it from the source files when it is not there. Nothing is lost if
you come back to this lab tomorrow.

In [ ]:
import matplotlib.pyplot as plt

def load_top10():
    """Read the output of Lab 1.1, or rebuild it from the source CSVs."""
    path = WORK / "top10.parquet"
    if path.exists():
        print("Reusing", path)
        return pd.read_parquet(path)
    print("Not found; rebuilding it from the source files.")
    kw = dict(sep=";", decimal=",", thousands=".", encoding="latin-1")
    counts = pd.read_csv(DATA / "aforos_202509.csv", parse_dates=["fecha"],
                         dayfirst=True, **kw).drop_duplicates()
    locations = pd.read_csv(DATA / "pm_ubicaciones.csv", **kw)
    means = (counts.groupby("id", as_index=False)["intensidad"].mean()
             .rename(columns={"intensidad": "mean_intensity"}))
    out = (means.sort_values("mean_intensity", ascending=False).head(10)
           .merge(locations, on="id", how="left").reset_index(drop=True))
    out.to_parquet(path, index=False)
    return out

top10 = load_top10()
print(len(top10), "measurement points carried over from Lab 1.1")

## 1. A shapefile is several files

A shapefile is not a file: it is a set. At least `.shp` (geometry), `.shx` (index) and `.dbf`
(attribute table, in 1983 dBase format). Usually also `.prj` (reference system) and `.cpg`
(text encoding). The last two are **optional**, and that is the problem.

In [ ]:
shp_dir = DATA / "municipios_shp"
for p in sorted(shp_dir.iterdir()):
    print(f"{p.name:<20} {p.stat().st_size:>8} bytes")

In [ ]:
municipalities = gpd.read_file(shp_dir / "municipios.shp")
print("CRS:", municipalities.crs)
print("Geometry types:", municipalities.geometry.geom_type.unique())
print("Columns:", list(municipalities.columns))
municipalities.head(3)

Look at the name of the last column. In the original file it was `superficie_km2`. The dBase
format limits field names to **10 characters**, so it was truncated on writing. This is the most
frequent cause of unreadable column names in official cartography.

### Exercise 1. Break the shapefile on purpose

Copy the shapefile's files to a `broken/` folder **without** the `.prj`, read it, and check what
happens to the reference system. Then repair it.

`set_crs()` declares a system the layer already had but had lost.
`to_crs()` transforms the coordinates to another system. Confusing them corrupts the data.

In [ ]:
import shutil
target = WORK / "broken"; target.mkdir(exist_ok=True)
for p in shp_dir.iterdir():
    if p.suffix != ".prj":
        shutil.copy(p, target / p.name)

# YOUR CODE HERE
raise NotImplementedError
print("CRS after losing the .prj:", lost_crs)
print("CRS after repairing:       ", repaired.crs)

In [ ]:
assert lost_crs is None, "Without a .prj the CRS should be None"
assert repaired.crs.to_epsg() == 25830
assert repaired.geometry.equals(municipalities.geometry), (
    "set_crs must not move any coordinate, only declare the system")
print("Checks passed.")

_Answer:_ if instead of `set_crs(25830)` you had used `to_crs(25830)` on the layer without a CRS, what would have happened?

### Exercise 2. Areas are not measured in degrees

The GeoJSON specification (RFC 7946) requires geographic coordinates in WGS 84, that is,
degrees, in longitude-latitude order. Read `municipios.geojson`, compute the area as it is and
compare it with the area computed after reprojecting to EPSG:25830 (metres).

The layer carries a `superficie_km2` column with the correct value, computed in a projected
system. Notice that here the name is **not** truncated: the 10-character cut is a limitation of
the shapefile, not of the data.

In [ ]:
geo = gpd.read_file(DATA / "municipios.geojson")
print("CRS of the GeoJSON:", geo.crs)

# YOUR CODE HERE
raise NotImplementedError

comparison = pd.DataFrame({
    "municipality": geo["nombre"],
    "area_in_deg2": area_deg.round(6),
    "area_km2_computed": area_km2.round(3),
    "area_km2_declared": geo["superficie_km2"].round(3),
})
comparison

In [ ]:
rel_error = ((comparison["area_km2_computed"] - comparison["area_km2_declared"]).abs()
             / comparison["area_km2_declared"]).max()
assert geo.crs.to_epsg() == 4326, "The GeoJSON must be in WGS 84"
assert geo_utm.crs.to_epsg() == 25830
assert rel_error < 0.01, f"Relative area error too large: {rel_error:.3%}"
print(f"Checks passed. Maximum relative error: {rel_error:.4%}")

> **Working rule.** `gdf.crs` first, `to_crs(25830)` second, `area` or `length` last.
> Never in the opposite order. The full treatment of reference systems is session 11.

### Exercise 3. From table to geographic table, and the CRS mismatch

`top10` is an ordinary table with two numeric columns that happen to be coordinates.
Turning it into a geographic table is one line. The problem comes afterwards.

**Step 1 (given).** Build the points **without declaring a reference system** and draw them over
the municipalities of the GeoJSON, which are in degrees. Before running, predict with your
partner: will the points land on the municipalities?

In [ ]:
points_nocrs = gpd.GeoDataFrame(
    top10, geometry=gpd.points_from_xy(top10["x_utm"], top10["y_utm"]))  # no crs

fig, ax = plt.subplots(figsize=(7, 5))
geo.plot(ax=ax, facecolor="#CFD8DC", edgecolor="white")
points_nocrs.plot(ax=ax, color="#E8590C", markersize=25)
ax.set_title("Municipalities in degrees, points in metres: two universes")
plt.tight_layout()
print("CRS of the municipalities:", geo.crs)
print("CRS of the points        :", points_nocrs.crs)

The municipalities sit near longitude -3.7 and latitude 40.4. The points, near x = 440 000 and
y = 4 480 000. No library complained: two things that do not share units were drawn on the same
plane.

**Step 2.** Repair with two different operations, in this order: `set_crs` **declares** the system
the coordinates already had (ETRS89 / UTM 30N); `to_crs` **transforms** to the system of the other
layer. Store the declared result in `points` and the transformed one in `points_4326`.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

fig, ax = plt.subplots(figsize=(7, 5))
geo.plot(ax=ax, facecolor="#CFD8DC", edgecolor="white")
points_4326.plot(ax=ax, color="#E8590C", markersize=25)
for _, r in points_4326.iterrows():
    ax.annotate(r["nombre"], (r.geometry.x, r.geometry.y), fontsize=7, xytext=(4, 4),
                textcoords="offset points")
ax.set_title("After set_crs(25830).to_crs(4326)")
plt.tight_layout()

In [ ]:
assert points_nocrs.crs is None, "Step 1 should have left the points without a CRS"
assert points.crs.to_epsg() == 25830, "set_crs should have declared UTM 30N"
assert points_4326.crs == geo.crs, "to_crs should have moved the points to the municipalities' system"
assert points.geometry.equals(points_nocrs.geometry), "set_crs must not move any coordinate"
minx, miny, maxx, maxy = geo.total_bounds
inside = points_4326.geometry.x.between(minx, maxx) & points_4326.geometry.y.between(miny, maxy)
assert inside.all(), "After transforming, the points should fall inside the municipalities' bounding box"
print("Checks passed.")

> **When two layers do not overlap, the first thing to check is the CRS of both.**
> The two operations are not interchangeable: `to_crs` on a layer without a declared CRS fails;
> `set_crs` with the wrong code moves the data to another continent without warning.

## 2. The field model

A municipality is an **object**: it has a boundary and attributes. Terrain elevation is a
**field**: it has a value at every point. Objects are stored in geographic tables; fields, in
surfaces.

### Exercise 4. Open the surface *(guided)*

This exercise is solved: run it and read the output. These are the five properties that define
any raster, and you will need them in the next exercise.

In [ ]:
src = rasterio.open(DATA / "mdt25_madrid.tif")
print("CRS       :", src.crs)
print("Resolution:", src.res, "m")
print("Extent    :", [round(v) for v in src.bounds])
print("Bands     :", src.count)
print("No-data   :", src.nodata)
print("Shape     :", src.shape, "->", src.shape[0] * src.shape[1], "cells")

In [ ]:
assert src.crs.to_epsg() == 25830
assert src.count == 1, "A digital terrain model has a single band"
assert src.nodata == -32768.0
assert src.res == (25.0, 25.0)
print("Checks passed.")

The no-data value is not `NaN`: it is an impossible number, just like the `-1` sentinel of
Lab 1.1. Drawn without masking, it shows up as an elevation of -32768 m that flattens the colour
scale. Check it.

In [ ]:
elev = src.read(1)
masked = np.where(elev == src.nodata, np.nan, elev)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
im0 = axes[0].imshow(elev, cmap="terrain"); axes[0].set_title("unmasked")
plt.colorbar(im0, ax=axes[0], shrink=.8)
im1 = axes[1].imshow(masked, cmap="terrain"); axes[1].set_title("no-data set to NaN")
plt.colorbar(im1, ax=axes[1], shrink=.8)
for a in axes: a.set_axis_off()
plt.tight_layout()
print("Range unmasked:", np.nanmin(elev).round(1), "to", np.nanmax(elev).round(1))
print("Range masked  :", np.nanmin(masked).round(1), "to", np.nanmax(masked).round(1))

### Exercise 5. The question of the session

Sample the surface at the ten points and add the column `elev_m`.
`src.sample()` takes a sequence of `(x, y)` pairs **in the raster's CRS** and returns a generator
of arrays, one per point.

Before sampling, check that the CRS of the points and that of the raster coincide.
If you did not, sampling would not raise an error: it would return values from somewhere else.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

answer = (points[["nombre", "distrito", "mean_intensity", "elev_m"]]
          .round({"mean_intensity": 1, "elev_m": 1}))
answer

In [ ]:
assert "elev_m" in points.columns
assert points["elev_m"].notna().all(), "Some point fell in a no-data area"
assert points["elev_m"].between(400, 1200).all(), (
    f"Elevations outside a plausible range: {points['elev_m'].min():.0f} to {points['elev_m'].max():.0f}")
print("Checks passed.")
print(f"\nAnswer to the question of the session:")
print(f"  mean elevation of the ten points: {points['elev_m'].mean():.1f} m")
print(f"  minimum {points['elev_m'].min():.1f} m, maximum {points['elev_m'].max():.1f} m")

### Exercise 6. Zonal statistics *(if time allows)*

A **zonal statistic** summarises a surface inside each object of a vector layer: the mean
elevation of each municipality, the accumulated rainfall of each catchment, the maximum slope of
each alignment section. It is the operation that connects the field model with the object model,
and you will use it throughout your career.

Compute the mean elevation of each municipality with `rasterio.mask.mask` and add it as a column.

In [ ]:
from rasterio.mask import mask

def mean_elev(geom, src):
    """Mean elevation inside a geometry, ignoring no-data cells."""
    # YOUR CODE HERE
    raise NotImplementedError

municipalities["mean_elev_m"] = [mean_elev(g, src) for g in municipalities.geometry]
municipalities[["nombre", "superficie", "mean_elev_m"]].round(1)

In [ ]:
assert municipalities["mean_elev_m"].notna().all()
assert municipalities["mean_elev_m"].between(400, 1200).all()
assert municipalities["mean_elev_m"].std() > 10, (
    "If all mean elevations are almost equal, the clipping is not working")
print("Checks passed.")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5))
municipalities.plot(column="mean_elev_m", cmap="terrain", legend=True, ax=ax,
                    edgecolor="white", linewidth=.8,
                    legend_kwds={"label": "mean elevation (m)", "shrink": .7})
points.plot(ax=ax, color="#E8590C", markersize=28, edgecolor="white", linewidth=.6)
ax.set_title("Mean elevation per municipality and traffic counters")
ax.set_axis_off(); plt.tight_layout()

### Extension A. One file, several layers

GeoPackage is a SQLite database with vector layers inside. Save municipalities and points as two
layers of the same `.gpkg`, list the layers and compare the size with the shapefile and the GeoJSON.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError

print(gpd.list_layers(gpkg_path))
size_shp = sum(p.stat().st_size for p in shp_dir.iterdir()) / 1024
print(f"\nShapefile (5 files)   {size_shp:7.1f} kB")
print(f"GeoJSON               {(DATA/'municipios.geojson').stat().st_size/1024:7.1f} kB")
print(f"GeoPackage (2 layers) {gpkg_path.stat().st_size/1024:7.1f} kB")

### Extension B. From surface to table

Rey, Arribas-Bel and Wolf (2023, ch. 5) discuss conversions between structures. Turn a small
window of the raster into a table with one row per cell and plot the elevation histogram.
What did you gain and what did you lose in the conversion?

In [ ]:
# YOUR CODE HERE
raise NotImplementedError


In [ ]:
src.close()
print("File closed.")

---

## Take-aways

- Object and field are different models stored in different structures: geographic table versus
  surface. The file format is secondary.
- A shapefile is three to six files; the one carrying the reference system is optional and
  easily lost.
- `set_crs` declares, `to_crs` transforms. They are not interchangeable, and when two layers do
  not overlap the first thing to check is the CRS of both.
- A raster's no-data value is a sentinel, with the same risks as the `-1` of the traffic table.
- Zonal statistics join the two models.

**References.** Rey, Arribas-Bel and Wolf (2023), ch. 3 and 5. ISO 19125-1 (Simple Features).
Lau, Gonzalez and Nolan (2023), ch. 8.